# 问题 6：单流派动态演化

深入研究一个流派的动态影响过程，识别动态影响者，解释流派/艺术家如何随时间变化。

**选择 Blues**（数据支持：艺术家数稳定、有大量有影响力的音乐人、文化背景强；可换成 Jazz / Pop/Rock）。

**内容：**
1. 流派艺术家出道年代分布
2. 影响来源流派（谁影响了 Blues）
3. 关键特征随年代轨迹
4. 各年代的主要影响者（动态影响者）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100

artists = pd.read_csv('output/artists.csv', dtype=str)
edges = pd.read_csv('output/edges.csv', dtype=str)
songs = pd.read_csv('output/songs.csv', dtype=str)
song_artist = pd.read_csv('output/song_artist.csv', dtype=str)

FEATURES = ['danceability', 'energy', 'valence', 'tempo', 'loudness',
            'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'duration_ms']

FOCUS = 'Blues'
f_artists = artists[artists['genre'] == FOCUS].copy()
f_artists['active_start'] = pd.to_numeric(f_artists['active_start'], errors='coerce')

print(f'{FOCUS} 艺术家数:', len(f_artists))

## 1. 出道年代分布

In [ ]:
start_counts = f_artists['active_start'].dropna().value_counts().sort_index()

plt.figure(figsize=(9, 4))
plt.bar(start_counts.index.astype(str), start_counts.values, color='#4C72B0')
plt.xlabel('Active start decade')
plt.ylabel('Number of artists')
plt.title(f'{FOCUS}: artists by active-start decade')
plt.tight_layout(); plt.show()

## 2. 影响来源流派

谁影响了 Blues 的艺术家（按影响者流派汇总）。

In [ ]:
blues_ids = set(f_artists['artist_id'])
blues_edges = edges[edges['follower_id'].isin(blues_ids)]
source_flow = blues_edges.groupby('influencer_main_genre').size().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(source_flow.index[::-1], source_flow.values[::-1], color='#55A868')
plt.xlabel('Number of influence edges')
plt.title(f'{FOCUS}: sources of influence (by influencer genre)')
plt.tight_layout(); plt.show()

## 3. 关键特征随年代轨迹

In [ ]:
genre_map = dict(zip(artists['artist_id'], artists['genre']))
sa = song_artist.copy()
sa['genre'] = sa['artist_id'].map(genre_map)

songs['year'] = pd.to_numeric(songs['year'], errors='coerce')
song_feat = songs[FEATURES + ['year']].copy()
song_feat['song_index'] = song_feat.index
for f in FEATURES:
    song_feat[f] = pd.to_numeric(song_feat[f], errors='coerce')

sa['song_index'] = sa['song_index'].astype(int)
sa = sa.merge(song_feat, on='song_index', how='left')
sa['decade'] = (sa['year'] // 10 * 10).astype('Int64')

focus_sa = sa[sa['genre'] == FOCUS].dropna(subset=['decade'])
trend = focus_sa.groupby('decade')[['energy', 'acousticness', 'instrumentalness']].mean()

fig, ax = plt.subplots(figsize=(10, 5))
for col in ['energy', 'acousticness', 'instrumentalness']:
    ax.plot(trend.index, trend[col], marker='o', label=col)
ax.set_xlabel('Decade')
ax.set_ylabel('Mean feature value')
ax.set_title(f'{FOCUS}: feature evolution over time')
ax.legend()
plt.tight_layout(); plt.show()

## 4. 各年代的主要影响者（动态影响者）

按追随者的出道年代分组，看每个年代被追随最多的影响者是谁。

In [ ]:
be = blues_edges.copy()
be['follower_active_start'] = pd.to_numeric(be['follower_active_start'], errors='coerce')
per_decade = (be.dropna(subset=['follower_active_start'])
              .groupby(['follower_active_start', 'influencer_name']).size()
              .reset_index(name='n'))
top_per_decade = (per_decade.sort_values('n', ascending=False)
                  .groupby('follower_active_start').head(3))

print(f'=== {FOCUS}: 各年代被追随最多的影响者 ===')
for d, grp in top_per_decade.groupby('follower_active_start'):
    names = ' / '.join(f'{r.influencer_name}({r.n})' for r in grp.itertuples())
    print(f'  {int(d)}s: {names}')

## 5. 结论

- 出道分布 + 影响来源说明流派的生命周期（兴起、鼎盛、衰落）。
- 特征轨迹（如 Blues 的 acousticness 下降、energy 上升）反映技术与社会变迁对该流派的影响。
- 各年代影响者名单揭示「谁在哪个阶段引领了流派方向」，即动态影响者。
- 注意：分时期影响力需对「积累时间」做调整，避免老艺术家仅因活跃久而天然占优。